# qlib 全流程研究工作流 Notebook

这本 notebook 用来把 `CSI300 + CSI500` 的 qlib 研究流程串成一条可重复执行的工作流水线。

设计原则：
1. 默认走正式版流程，不默认走 smoke。
2. notebook 只编排流程，不重复实现核心业务逻辑。
3. 训练、评估和发布都复用仓库里的同一套 helper。
4. 每一步都明确给出“看什么、怎么判断、下一步做什么”。


## 1. 目标与使用方式

本 notebook 适合在以下场景使用：

1. 从零开始准备 `merged_csi300_500` 的研究数据。
2. 跑 `full_postfix_baseline -> small_stable` 的正式收敛评估。
3. 对比主池、`CSI300`、`CSI500` 三个层面的表现。
4. 判断这次结果是“保留研究结论”还是“进入候选发布”。

建议使用方式：

1. 第一次跑先保持 `SMOKE_MODE = False`，只在数据量或环境异常时切换 smoke。
2. 每次正式跑都使用新的 `OUTPUT_DIR`，避免覆盖历史评估。
3. 只有当 `promotion_gate_passed = true` 且你确认结果稳定时，才打开最后一节的发布开关。


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from qlib_research.core.notebook_workflow import (
    ensure_runtime,
    export_or_load_panel,
    load_evaluation_artifacts,
    publish_from_selected_recipe,
    run_convergence_workflow,
    summarize_gate_decision,
)
from qlib_research.core.qlib_pipeline import FEATURE_GROUP_COLUMNS, FULL_POSTFIX_BASELINE_FEATURE_COLUMNS
from qlib_research.core.weekly_model_eval import prefilter_feature_columns

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

display(Markdown(f"**PROJECT_ROOT**: `{PROJECT_ROOT}`"))

def _format_metric(value: object, digits: int = 4) -> str:
    if value is None or pd.isna(value):
        return "NA"
    return f"{float(value):,.{digits}f}"

def render_stage_summary(stage_summary: pd.DataFrame) -> None:
    if stage_summary.empty:
        display(Markdown("**阶段摘要**：当前没有可展示的阶段摘要。"))
        return
    columns = [
        "stage",
        "recipe",
        "feature_count",
        "eval_date_count",
        "tuning_candidate_count",
        "accepted_group_prunes",
        "accepted_feature_prunes",
        "rank_ic_ir",
        "topk_mean_excess_return_4w",
        "strategy_max_drawdown",
        "promotion_gate_passed",
        "stage_elapsed_human",
    ]
    available_columns = [column for column in columns if column in stage_summary.columns]
    display(stage_summary[available_columns])
    cards = []
    for _, row in stage_summary.iterrows():
        gate_text = "通过" if bool(row.get("promotion_gate_passed", False)) else "研究中/未通过"
        cards.append(
            "<div style='border:1px solid #d0d7de;border-radius:12px;padding:12px 14px;margin:8px 0;background:#fafbfc;'>"
            f"<div style='font-weight:600;font-size:15px;margin-bottom:6px'>{row.get('stage_label', row.get('stage', 'unknown'))}</div>"
            f"<div>recipe: <code>{row.get('recipe', 'NA')}</code></div>"
            f"<div>features: <b>{int(row.get('feature_count', 0) or 0)}</b> | eval_dates: <b>{int(row.get('eval_date_count', 0) or 0)}</b></div>"
            f"<div>rank_ic_ir: <b>{_format_metric(row.get('rank_ic_ir'))}</b> | topk_excess: <b>{_format_metric(row.get('topk_mean_excess_return_4w'))}</b></div>"
            f"<div>drawdown: <b>{_format_metric(row.get('strategy_max_drawdown'))}</b> | gate: <b>{gate_text}</b></div>"
            f"<div>elapsed: <b>{row.get('stage_elapsed_human', 'NA')}</b></div>"
            + "</div>"
        )
    display(Markdown("\n".join(cards)))


**PROJECT_ROOT**: `/Volumes/Repository/Projects/TradingNexus/QlibResearch`

## 2. 环境检查与依赖确认

这一节先确认 notebook 当前环境是否具备正式执行条件。

判断依据：

1. `pyqlib`、`plotly`、`ipykernel`、`finance-data-hub` 是否可用。
2. 当前工作目录是否指向项目根目录。
3. 目标 panel 路径是否已存在。

如果 `pyqlib_version` 为空，说明当前环境还不能跑训练和评估，需要先补依赖。


In [2]:
runtime_info = ensure_runtime(panel_path="artifacts/panels/csi300500_weekly.csv")
runtime_frame = pd.DataFrame([runtime_info]).T.rename(columns={0: "value"})
display(runtime_frame)

assert runtime_info["pyqlib_version"], "未检测到 pyqlib，请先安装 qlib 依赖后再执行后续单元。"
assert runtime_info["finance_data_hub_version"], "未检测到 finance-data-hub，请先确认本地依赖和 sources.yml。"


,value
project_root,/Volumes/Repository/Projects/TradingNexus/Valu...
cwd,/Volumes/Repository/Projects/TradingNexus/Valu...
python_version,3.12.10
artifacts_dir,/Volumes/Repository/Projects/TradingNexus/Valu...
ipykernel_version,7.2.0
plotly_version,6.5.1
finance_data_hub_version,0.1.0
pyqlib_version,0.9.7
panel_path,/Volumes/Repository/Projects/TradingNexus/Valu...
panel_exists,False


## 3. 参数区

这一区域集中管理本次研究参数。

判断依据：

1. 正式评估默认使用 `merged_csi300_500`、`final_compare` 和 `l1_weekly_robust`。
2. notebook 默认使用“受控收敛”参数，而不是不设上限的穷举删减，避免 `feature_prune` 直接膨胀到数万次 fit。
3. `RUN_EXPORT = "auto_if_missing"` 表示 panel 不存在时自动导出，存在时直接复用。
4. `CONSOLE_MODE = "bar"` 默认使用单行进度条；如果 notebook 前端不适合动态刷新，可以切到 `silent`。
5. `RUN_PUBLISH = False` 是默认安全值，避免 notebook 被误点后直接发布模型。
6. `SMOKE_MODE = True` 只用于排障，不用于策略判断。


In [3]:
DATE_TAG = pd.Timestamp.now(tz="Asia/Shanghai").strftime("%Y%m%d")

UNIVERSE_PROFILE = "merged_csi300_500"
PANEL_PATH = "artifacts/panels/csi300500_weekly.csv"
OUTPUT_DIR = f"artifacts/evaluations/csi300500_notebook_controlled_{DATE_TAG}"
SELECTION_MODE = "final_compare"
INDUSTRY_NORMALIZATION = "l1_weekly_robust"
EVAL_COUNT = 52
TRAIN_WEEKS = 260
VALID_WEEKS = 52
TOPK = 10
RUN_EXPORT = "auto_if_missing"
RUN_PUBLISH = False
CONSOLE_MODE = "bar"
SMOKE_MODE = False

START_DATE = None
END_DATE = None
STEP_WEEKS = 1
BATCH_SIZE = 300
COARSE_TUNE_LIMIT = 6
MICRO_TUNE_LIMIT = 3
MAX_GROUP_PRUNE_ROUNDS = 2
GROUP_CANDIDATE_LIMIT = 4
MAX_FEATURE_PRUNE_ROUNDS = 3
FEATURE_CANDIDATE_LIMIT = 8
PUBLISH_MODEL_ID = None
PUBLISH_EXPERIMENT_NAME = "valueinvesting_weekly_qlib_notebook"

if SMOKE_MODE:
    PANEL_PATH = "artifacts/panels/csi300500_tiny_smoke.csv"
    OUTPUT_DIR = f"artifacts/evaluations/csi300500_notebook_smoke_{DATE_TAG}"
    EVAL_COUNT = 1
    TRAIN_WEEKS = 10
    VALID_WEEKS = 4
    TOPK = 5
    COARSE_TUNE_LIMIT = 1
    MICRO_TUNE_LIMIT = 0
    MAX_GROUP_PRUNE_ROUNDS = 1
    GROUP_CANDIDATE_LIMIT = 1
    MAX_FEATURE_PRUNE_ROUNDS = 1
    FEATURE_CANDIDATE_LIMIT = 1

parameter_frame = pd.DataFrame(
    [
        ("UNIVERSE_PROFILE", UNIVERSE_PROFILE),
        ("PANEL_PATH", PANEL_PATH),
        ("OUTPUT_DIR", OUTPUT_DIR),
        ("SELECTION_MODE", SELECTION_MODE),
        ("INDUSTRY_NORMALIZATION", INDUSTRY_NORMALIZATION),
        ("EVAL_COUNT", EVAL_COUNT),
        ("TRAIN_WEEKS", TRAIN_WEEKS),
        ("VALID_WEEKS", VALID_WEEKS),
        ("TOPK", TOPK),
        ("COARSE_TUNE_LIMIT", COARSE_TUNE_LIMIT),
        ("MICRO_TUNE_LIMIT", MICRO_TUNE_LIMIT),
        ("MAX_GROUP_PRUNE_ROUNDS", MAX_GROUP_PRUNE_ROUNDS),
        ("GROUP_CANDIDATE_LIMIT", GROUP_CANDIDATE_LIMIT),
        ("MAX_FEATURE_PRUNE_ROUNDS", MAX_FEATURE_PRUNE_ROUNDS),
        ("FEATURE_CANDIDATE_LIMIT", FEATURE_CANDIDATE_LIMIT),
        ("RUN_EXPORT", RUN_EXPORT),
        ("RUN_PUBLISH", RUN_PUBLISH),
        ("CONSOLE_MODE", CONSOLE_MODE),
        ("SMOKE_MODE", SMOKE_MODE),
    ],
    columns=["parameter", "value"],
)
display(parameter_frame)


,parameter,value
0,UNIVERSE_PROFILE,merged_csi300_500
1,PANEL_PATH,artifacts/panels/csi300500_weekly.csv
2,OUTPUT_DIR,artifacts/evaluations/csi300500_note...
3,SELECTION_MODE,final_compare
4,INDUSTRY_NORMALIZATION,l1_weekly_robust
5,EVAL_COUNT,52
6,TRAIN_WEEKS,260
7,VALID_WEEKS,52
8,TOPK,10
9,COARSE_TUNE_LIMIT,6


## 4. 数据导出与 panel 生成

这一节负责准备 `csi300500_weekly.csv`。

判断依据：

1. 导出后先看起止日期和样本股票数是否合理。
2. 必须确认 `in_csi300` 和 `in_csi500` 字段存在。
3. 如果 panel 已存在且你确认它是最新的，`RUN_EXPORT = "auto_if_missing"` 会直接复用。


In [4]:
panel_result = export_or_load_panel(
    panel_path=PANEL_PATH,
    universe_profile=UNIVERSE_PROFILE,
    run_export=RUN_EXPORT,
    start_date=START_DATE,
    end_date=END_DATE,
    batch_size=BATCH_SIZE,
    return_panel=True,
)
panel = panel_result["panel"]

display(pd.DataFrame([{"action": panel_result["action"], **panel_result["summary"]}]))

required_columns = {"datetime", "instrument", "in_csi300", "in_csi500"}
missing_columns = required_columns - set(panel.columns)
assert not missing_columns, f"panel 缺少必要字段: {sorted(missing_columns)}"


2026-04-01 19:52:13.660 | INFO     | finance_data_hub.providers.registry:register:52 - Registered provider: tushare -> TushareProvider
2026-04-01 19:52:13.723 | INFO     | finance_data_hub.providers.registry:register:52 - Registered provider: xtquant -> XTQuantProvider
2026-04-01 19:52:13.744 | DEBUG    | finance_data_hub.preprocessing.technical.base:register:130 - Registered indicator: ma_20
2026-04-01 19:52:13.745 | DEBUG    | finance_data_hub.preprocessing.technical.base:register:130 - Registered indicator: ema_20
2026-04-01 19:52:13.745 | DEBUG    | finance_data_hub.preprocessing.technical.base:register:130 - Registered indicator: ma_50
2026-04-01 19:52:13.745 | DEBUG    | finance_data_hub.preprocessing.technical.base:register:130 - Registered indicator: ema_50
2026-04-01 19:52:13.747 | DEBUG    | finance_data_hub.preprocessing.technical.base:register:130 - Registered indicator: macd
2026-04-01 19:52:13.747 | DEBUG    | finance_data_hub.preprocessing.technical.base:register:130 - R

,action,rows,instrument_count,start_date,end_date,csi300_rows,csi500_rows,csi300_instruments,csi500_instruments
0,exported,602002,1839,2007-02-02,2026-04-03,150652,451350,607,1652


## 5. panel 质量检查与 EDA

先看数据是否覆盖了我们预期的 universe 和时间段。

判断依据：

1. `rows / instrument_count / start_date / end_date` 是否符合预期。
2. `CSI300 / CSI500` 的样本量是否明显失衡。
3. 核心字段是否存在明显缺失或时间断层。


In [5]:
summary_frame = pd.DataFrame([panel_result["summary"]])
display(summary_frame)

preview_columns = [
    column
    for column in [
        "datetime", "instrument", "in_csi300", "in_csi500", "l1_name", "close",
        "pe_ttm", "f_score", "label_excess_return_4w"
    ]
    if column in panel.columns
]
display(panel[preview_columns].head(10))

coverage_by_date = (
    panel.groupby("datetime")
    .agg(
        rows=("instrument", "size"),
        instruments=("instrument", "nunique"),
        csi300_rows=("in_csi300", "sum"),
        csi500_rows=("in_csi500", "sum"),
    )
    .reset_index()
)

coverage_fig = px.line(
    coverage_by_date,
    x="datetime",
    y=["rows", "instruments", "csi300_rows", "csi500_rows"],
    title="panel 覆盖概览",
    markers=False,
)
coverage_fig.update_layout(height=500)
coverage_fig.show()


,rows,instrument_count,start_date,end_date,csi300_rows,csi500_rows,csi300_instruments,csi500_instruments
0,602002,1839,2007-02-02,2026-04-03,150652,451350,607,1652


,datetime,instrument,in_csi300,in_csi500,l1_name,close,pe_ttm,f_score,label_excess_return_4w
0,2016-01-29,000001.SZ,True,False,银行,6.389834,6.549200,6.000000,-0.009337
1,2016-02-05,000001.SZ,True,False,银行,6.338715,6.496800,6.000000,0.031369
2,2016-02-19,000001.SZ,True,False,银行,6.415393,6.575400,6.000000,0.034693
3,2016-02-26,000001.SZ,True,False,银行,6.255647,6.411700,6.000000,-0.015403
4,2016-03-04,000001.SZ,True,False,银行,6.645427,6.811200,6.000000,-0.092765
5,2016-03-11,000001.SZ,True,False,银行,6.492071,6.648800,6.000000,-0.101077
6,2016-03-18,000001.SZ,True,False,银行,6.734885,6.897500,6.000000,-0.055161
7,2016-03-25,000001.SZ,True,False,银行,6.766834,6.930200,6.000000,-0.004091
8,2016-04-01,000001.SZ,True,False,银行,6.811563,6.976000,6.000000,-0.000392
9,2016-04-08,000001.SZ,True,False,银行,6.754054,6.917100,6.000000,0.015539


## 6. 特征组、行业标准化与预过滤

这一节先在 panel 层看特征质量，不做训练。

判断依据：

1. 缺失率高于 `35%` 的特征会被剔除。
2. 长期近似常数或横截面波动过低的特征会被剔除。
3. 高相关特征不会立刻删除，但会被记录到候选冗余清单中。
4. 行业内标准化只作用于估值和质量类特征，技术特征不做行业内改写。


In [6]:
feature_group_frame = pd.DataFrame(
    [
        {
            "feature_group": group_name,
            "feature_count": len(columns),
            "features": ", ".join(columns),
        }
        for group_name, columns in FEATURE_GROUP_COLUMNS.items()
    ]
)
display(feature_group_frame)

filtered_features, feature_stats, corr_marks = prefilter_feature_columns(panel, FULL_POSTFIX_BASELINE_FEATURE_COLUMNS)
display(pd.DataFrame([
    {
        "full_feature_count": len(FULL_POSTFIX_BASELINE_FEATURE_COLUMNS),
        "filtered_feature_count": len(filtered_features),
        "removed_feature_count": len(FULL_POSTFIX_BASELINE_FEATURE_COLUMNS) - len(filtered_features),
    }
]))
display(feature_stats.head(20))

missing_fig = px.bar(
    feature_stats.sort_values("missing_ratio", ascending=False),
    x="feature",
    y="missing_ratio",
    color="keep",
    title="特征缺失率与保留结果",
)
missing_fig.update_layout(height=500, xaxis_tickangle=-45)
missing_fig.show()

if not corr_marks.empty:
    display(corr_marks.head(20))
    heatmap_features = list(dict.fromkeys(corr_marks["left_feature"].head(6).tolist() + corr_marks["right_feature"].head(6).tolist()))
    corr_source = panel[heatmap_features].apply(pd.to_numeric, errors="coerce")
    corr_source = corr_source.fillna(corr_source.median(numeric_only=True))
    corr_matrix = corr_source.corr()
    corr_fig = px.imshow(corr_matrix, text_auto=".2f", aspect="auto", title="高相关候选热图（截取）")
    corr_fig.update_layout(height=500)
    corr_fig.show()
else:
    display(Markdown("**高相关候选**：当前预过滤阶段没有发现超过阈值的高相关特征。"))


,feature_group,feature_count,features
0,technical_core,11,"signal_strength, signal_grade_num, ma20, ma50,..."
1,valuation_absolute,5,"pe_ttm, pb, ps_ttm, dv_ttm, peg"
2,valuation_percentile,3,"pe_ttm_pct_1250d, pb_pct_1250d, ps_ttm_pct_1250d"
3,industry_valuation_context,2,"core_indicator_pct_1250d, core_indicator_indus..."
4,quality_summary,5,"f_score, roe_5y_avg, ni_cfo_corr_3y, debt_rati..."
5,fscore_components,9,"f_roa, f_cfo, f_delta_roa, f_accrual, f_delta_..."
6,ttm_profitability,5,"roa_ttm, cfo_ttm, ni_ttm, gpm_ttm, at_ttm"


,full_feature_count,filtered_feature_count,removed_feature_count
0,40,36,4


,feature,missing_ratio,overall_std,median_cross_section_std,keep
0,amount,0.000000,"3,818,955.980886","1,554,664.029957",True
1,volume,0.000000,"2,635,829.689985","1,398,036.070235",True
2,atr,0.022686,3.790050,1.042236,True
3,rsi,0.022686,11.985993,8.728957,True
4,ma20,0.028965,47.482161,10.221225,True
5,dea,0.048075,2.898918,0.746535,True
6,dif,0.048075,3.091101,0.789369,True
7,macd_hist,0.048075,1.871073,0.489700,True
8,ma50,0.074028,46.637400,8.270535,True
9,f_accrual,0.137732,0.473722,0.477076,True


,left_feature,right_feature,abs_corr
0,ma20,ma50,0.980573
1,dif,dea,0.953229
2,ma20,atr,0.913570


## 7. `full_postfix_baseline` 配置与正式评估

这一节会真正触发正式收敛流程，是全 notebook 最重的一步。

判断依据：

1. 正式版默认跑 `final_compare`，但参数区默认已经收紧为“受控收敛”版本。
2. 如果你把 `*_LIMIT` 和 `MAX_*_PRUNE_ROUNDS` 全部放开，`feature_prune` 的 fit 数量会非常快地膨胀。
3. 主池、`CSI300`、`CSI500` 三套结果会一起产出。
4. 评估产物会落盘到 `OUTPUT_DIR`，后续章节全部以这里的结果为准。


In [ ]:
workflow_result = run_convergence_workflow(
    panel_or_path=panel,
    output_dir=OUTPUT_DIR,
    universe_profile=UNIVERSE_PROFILE,
    selection_mode=SELECTION_MODE,
    industry_normalization=INDUSTRY_NORMALIZATION,
    emit_slices=True,
    eval_count=EVAL_COUNT,
    step_weeks=STEP_WEEKS,
    train_weeks=TRAIN_WEEKS,
    valid_weeks=VALID_WEEKS,
    topk=TOPK,
    start_date=START_DATE,
    end_date=END_DATE,
    experiment_name="valueinvesting_weekly_qlib_notebook",
    coarse_tune_limit=COARSE_TUNE_LIMIT,
    micro_tune_limit=MICRO_TUNE_LIMIT,
    max_group_prune_rounds=MAX_GROUP_PRUNE_ROUNDS,
    group_candidate_limit=GROUP_CANDIDATE_LIMIT,
    max_feature_prune_rounds=MAX_FEATURE_PRUNE_ROUNDS,
    feature_candidate_limit=FEATURE_CANDIDATE_LIMIT,
    console_mode=CONSOLE_MODE,
)

display(workflow_result["summary"])


## 8. 分组回退删减与单特征删减

这一节不再重新训练，而是读取上一节已经产出的删减日志。

判断依据：

1. `feature_group_prune` 看的是哪个特征组被尝试删除、是否被接受。
2. `feature_prune_log` 看的是单特征删减路径。
3. notebook 默认只展示“受控候选集”下的删减结果，不等于做了全量穷举。
4. 如果这里几乎没有删减成功，通常意味着当前全量特征还不够冗余，或当前 gate 太严格。


In [ ]:
group_prune_log = workflow_result["group_prune_log"]
feature_prune_log = pd.DataFrame(workflow_result["feature_prune_log"])

if group_prune_log.empty:
    display(Markdown("**分组删减结果**：当前流程没有产生可展示的分组删减记录。"))
else:
    display(group_prune_log.head(20))

if feature_prune_log.empty:
    display(Markdown("**单特征删减结果**：当前流程没有产生可展示的单特征删减记录。"))
else:
    display(feature_prune_log.head(20))


## 9. `final_compare` 结果汇总与晋级判断

这一节以落盘产物为准，不直接依赖内存对象。

判断依据：

1. `summary.csv` 看主池整体表现。
2. `slice_summary.csv` 看 `CSI300` 与 `CSI500` 切片是否同步站住。
3. `comparison.json` 看晋级门槛每一项是否通过。
4. 若任一关键门槛未通过，默认结论就是“保留研究结论，不发布”。


In [ ]:
artifacts = load_evaluation_artifacts(OUTPUT_DIR)
gate_summary = summarize_gate_decision(artifacts)
stage_summary = artifacts["stage_summary"].copy()
latest_stage = stage_summary.iloc[-1].to_dict() if not stage_summary.empty else {}

display(Markdown("**阶段进度表与阶段卡片**"))
render_stage_summary(stage_summary)

display(Markdown("**核心结果表**"))
display(artifacts["summary"])
display(artifacts["slice_summary"])

comparison_frame = pd.DataFrame([artifacts["comparison"].get("comparison", {})])
promotion_gate_frame = pd.DataFrame([artifacts["comparison"].get("promotion_gate", {})]).T.rename(columns={0: "value"})
selected_recipe_frame = pd.DataFrame([artifacts["selected_recipe"]])

display(comparison_frame)
display(promotion_gate_frame)
display(selected_recipe_frame)
display(pd.DataFrame([gate_summary]))

status_text = "可以进入候选发布" if gate_summary["promotion_gate_passed"] else "不建议发布，保留研究结论"
blockers = "<br>".join(f"- {item}" for item in gate_summary["blocking_reasons"]) or "- 当前没有阻塞项"
stage_note = (
    f"当前阶段：`{latest_stage.get('stage_label', latest_stage.get('stage', 'NA'))}`，"
    f"特征数 `{int(latest_stage.get('feature_count', 0) or 0)}`，"
    f"RankIC_IR `{_format_metric(latest_stage.get('rank_ic_ir'))}`，"
    f"TopK 超额 `{_format_metric(latest_stage.get('topk_mean_excess_return_4w'))}`。"
    if latest_stage
    else "当前没有阶段摘要。"
)
display(Markdown(f"**晋级结论**：{status_text}<br><br>**阶段摘要**：{stage_note}<br><br>**阻塞项**：<br>{blockers}"))


## 10. 预测、滚动回测与切片分析

这一节主要看三类图：

1. `rank_ic` 的时间序列，判断排序稳定性。
2. `topk_mean_excess_return_4w` 的时间序列，判断组合超额收益。
3. `all / csi300 / csi500` 的净值曲线，判断回撤和收益分布。


In [ ]:

details = artifacts["details"]
predictions = artifacts["predictions"]
equity_curve = artifacts["equity_curve"]

if not details.empty:
    details_plot = details.melt(
        id_vars=["feature_date", "recipe"],
        value_vars=["rank_ic", "topk_mean_excess_return_4w"],
        var_name="metric",
        value_name="value",
    )
    details_fig = px.line(
        details_plot,
        x="feature_date",
        y="value",
        color="metric",
        facet_row="recipe",
        title="RankIC 与 TopK 超额收益时间序列",
    )
    details_fig.update_layout(height=700)
    details_fig.show()

if not equity_curve.empty:
    equity_plot = equity_curve.copy()
    equity_plot["series"] = equity_plot["recipe"].astype(str) + " | " + equity_plot["slice"].astype(str)
    equity_fig = px.line(
        equity_plot,
        x="datetime",
        y="total_value",
        color="series",
        title="主池与切片净值曲线",
    )
    equity_fig.update_layout(height=550)
    equity_fig.show()

if not predictions.empty:
    latest_prediction_date = pd.to_datetime(predictions["feature_date"]).max()
    latest_prediction = predictions.loc[pd.to_datetime(predictions["feature_date"]) == latest_prediction_date].copy()
    latest_prediction = latest_prediction.sort_values("score", ascending=False)
    display(Markdown(f"**最新预测日期**：`{latest_prediction_date.date()}`"))
    display(latest_prediction[[column for column in ["recipe", "instrument", "score", "future_return_4w", "l1_name", "in_csi300", "in_csi500"] if column in latest_prediction.columns]].head(20))


## 11. 结果解读与下一步动作

这一节的目标不是再算新指标，而是把当前结果转成明确动作。

判断依据：

1. 若 `promotion_gate_passed = false`，默认动作是保留研究结论、不发布。
2. 若门槛通过，再决定是否进入可选发布节点。
3. 即使门槛通过，也建议先人工查看 `summary.csv / slice_summary.csv / comparison.json` 三份核心产物。


In [ ]:

final_stage = stage_summary.iloc[-1].to_dict() if not stage_summary.empty else {}
stage_guidance = (
    f"最后阶段 `{final_stage.get('stage_label', final_stage.get('stage', 'NA'))}` 的 RankIC_IR 为 `{_format_metric(final_stage.get('rank_ic_ir'))}`，"
    f"TopK 超额为 `{_format_metric(final_stage.get('topk_mean_excess_return_4w'))}`，"
    f"最大回撤为 `{_format_metric(final_stage.get('strategy_max_drawdown'))}`。"
    if final_stage
    else "当前没有阶段摘要，建议先检查 stage_summary.csv 是否已生成。"
)

if gate_summary["promotion_gate_passed"]:
    guidance = [
        "本次评估通过晋级门槛，可以进入候选发布流程。",
        stage_guidance,
        "发布前仍建议人工复核主池与切片表现，并确认输出目录命名可追踪。",
        "如果要发布，建议保留当前 OUTPUT_DIR 作为本次研究快照。",
    ]
else:
    guidance = [
        "本次评估未通过晋级门槛，建议保留研究结论，不发布线上候选模型。",
        stage_guidance,
        "优先检查 gate 中未通过的项，尤其是 rank、slice 和 reduction 三类条件。",
        "如果要继续研究，建议复制当前参数区，形成下一次实验的对照版本。",
    ] + gate_summary["blocking_reasons"]

guidance_markdown = "\n".join(
    f"1. {text}" if index == 0 else f"{index + 1}. {text}"
    for index, text in enumerate(guidance)
)
display(Markdown(guidance_markdown))


## 12. 可选发布线上候选模型

这一节默认不会执行发布。

判断依据：

1. 只有当 `RUN_PUBLISH = True` 且 `promotion_gate_passed = true` 时，才会真正调用训练发布。
2. 如果门槛未通过，本单元会直接阻止发布。
3. 发布使用的特征配置来自 `selected_recipe.json`，不会绕过收敛结果。


In [ ]:
selected_recipe_path = Path(OUTPUT_DIR) / "selected_recipe.json"
selected_recipe = artifacts["selected_recipe"]
candidate_model_id = PUBLISH_MODEL_ID or selected_recipe.get("model_id_suggestion") or f"csi300500-weekly-lgbm-small-stable-{DATE_TAG}"

display(pd.DataFrame([
    {
        "RUN_PUBLISH": RUN_PUBLISH,
        "promotion_gate_passed": gate_summary["promotion_gate_passed"],
        "selected_recipe_path": str(selected_recipe_path),
        "candidate_model_id": candidate_model_id,
    }
]))

if RUN_PUBLISH and not gate_summary["promotion_gate_passed"]:
    raise RuntimeError("本次评估未通过晋级门槛，不建议发布线上候选模型。")

publish_result = publish_from_selected_recipe(
    panel_path=PANEL_PATH,
    feature_config_path=selected_recipe_path,
    model_id=candidate_model_id,
    experiment_name=PUBLISH_EXPERIMENT_NAME,
    run_publish=RUN_PUBLISH,
)
display(pd.DataFrame([publish_result]))
